# Random Digraph Synchronization Simulation

Simulates a network of Lorenz oscillators coupled over a random directed graph
that contains exactly one "root" strongly-connected component reaching every
other node. Plots are rendered **inline** in this notebook instead of being
saved to disk.

In [ ]:
# Colab already ships with numpy, scipy, pandas, matplotlib, networkx.
# openpyxl is needed for the .xlsx export at the end.
!pip install -q openpyxl

In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

%matplotlib inline
plt.close("all")

## 1) Lorenz Oscillator (vectorized over all N nodes at once)

In [ ]:
def lorenz_oscillator(t, U, sigma=10.0, rho=28.0, beta=8 / 3):
    """
    Uncoupled Lorenz dynamics for every node at once.
    U has shape (state_dim, N) -- one column per node.
    """
    dU = np.empty_like(U)
    dU[0, :] = sigma * (U[1, :] - U[0, :])
    dU[1, :] = U[0, :] * (rho - U[2, :]) - U[1, :]
    dU[2, :] = U[0, :] * U[1, :] - beta * U[2, :]
    return dU

## 2) Coupled Network Dynamics

In [ ]:
def coupled_dynamics(t, u, system_dynamics, L, P, a, N, state_dim):
    """
    Right-hand side of the coupled network ODE, flattened to a 1-D
    vector for scipy's solve_ivp. Node i's state occupies the block
    u[i*state_dim : (i+1)*state_dim] (matches Julia's column-major
    reshape(u, state_dim, N) convention).
    """
    U = u.reshape((state_dim, N), order="F")

    dU = system_dynamics(t, U)          # each node's own dynamics
    PU = P @ U
    dU = dU - PU @ L.T - a * PU         # diffusive coupling + pinning term

    return dU.reshape(-1, order="F")

## 3) Simulation Wrapper

In [ ]:
def simulate_coupled_systems(system_dynamics, tspan, X0, G, P, a):
    n = G.number_of_nodes()

    # In-degree adjacency convention: A[dst, src] = weight
    A = np.zeros((n, n))
    for u, v, data in G.edges(data=True):
        A[v, u] = data.get("weight", 1.0)

    D = np.diag(A.sum(axis=1))
    L = D - A.T

    state_dim = P.shape[0]

    sol = solve_ivp(
        coupled_dynamics,
        (tspan[0], tspan[-1]),
        X0,
        method="RK45",          # closest scipy equivalent to Julia's Tsit5
        t_eval=tspan,
        args=(system_dynamics, L, P, a, n, state_dim),
        rtol=1e-3,
        atol=1e-6,
    )

    X = sol.y.T   # rows = time points, columns = flattened state
    t = sol.t
    return X, t, A

## 4) Generate Directed Graph with ONE Root SCC

In [ ]:
def generate_one_root_scc(n, p=None):
    """
    Builds a random digraph on n nodes (0-indexed) that contains
    exactly one strongly connected "root" component (a directed
    cycle) which reaches every other vertex.
    """
    if p is None:
        p = 0.2 + 0.6 * np.random.random()   # uniform draw

    G = nx.DiGraph()
    G.add_nodes_from(range(n))

    # --- Step 1: choose size of root SCC ---
    k = np.random.randint(1, n + 1)

    root_vertices = np.arange(k)
    other_vertices = np.arange(k, n)

    # --- Step 2: strongly connected root SCC (a cycle) ---
    if k > 1:
        for i in range(k):
            G.add_edge(i, (i + 1) % k)

    # --- Step 3: random digraph on the remaining vertices ---
    for i in other_vertices:
        for j in other_vertices:
            if i != j and np.random.random() < p:
                G.add_edge(int(i), int(j))

    # --- Step 4: force edges from root SCC -> others ---
    for v in other_vertices:
        G.add_edge(int(np.random.choice(root_vertices)), int(v))

    return G

## 5) Assign Coupling Weights

In [ ]:
def sync_coupling_assign(G, weight):
    for u, v in G.edges():
        G[u][v]["weight"] = weight
    return G

## Run the Simulation

In [ ]:
print("🚀 Random Digraph with Exactly One Root SCC")
print("============================================")

N = 50

sigma, rho, beta = 10.0, 25.0, 8 / 3
a = -sigma + (beta * (beta + 1) * (rho + sigma) ** 2) / (16 * (beta - 1))

print(f"🔧 Coupling a = {round(a, 4)}")

G = generate_one_root_scc(N)
G = sync_coupling_assign(G, 1.01 * a)

### Figure 1 — Network Plot

In [ ]:
theta = np.linspace(0, 2 * np.pi, N, endpoint=False)
x = np.cos(theta)
y = np.sin(theta)

fig1, ax1 = plt.subplots(figsize=(6, 6))
ax1.scatter(x, y, s=60, c="lightblue", zorder=3)

for u, v in G.edges():
    ax1.annotate(
        "",
        xy=(x[v], y[v]),
        xytext=(x[u], y[u]),
        arrowprops=dict(arrowstyle="-|>", color="black", lw=1.2,
                         shrinkA=6, shrinkB=6),
    )

ax1.set_aspect("equal")
ax1.axis("off")
plt.show()

### Figure 2 — Pairwise Distances

In [ ]:
P = np.diag([1.0, 0.0, 0.0])
tspan = np.linspace(0, 10, 200)
X0 = 1e-4 * np.random.randn(3 * N)

print("⏱️ Simulating...")
X, t, A = simulate_coupled_systems(lorenz_oscillator, tspan, X0, G, P, a)

In [ ]:
print("📏 Computing pairwise distances...")
num_time = X.shape[0]
pairwise_distances = np.zeros((num_time, N - 1))

for k_ in range(num_time):
    U = X[k_, :].reshape((3, N), order="F")
    for i in range(N - 1):
        pairwise_distances[k_, i] = np.linalg.norm(U[:, i] - U[:, i + 1])

In [ ]:
xmax = t.max()
ymax = pairwise_distances.max()
y_upper = ymax * 1.08

fig2, ax2 = plt.subplots(figsize=(10, 6))
for i in range(N - 1):
    ax2.plot(t, pairwise_distances[:, i], lw=1.5)

ax2.scatter(
    np.zeros(N - 1),
    pairwise_distances[0, :],
    color="black",
    s=16,
    linewidths=0.5,
    zorder=3,
)

ax2.set_xlim(0, xmax)
ax2.set_ylim(0, y_upper)
ax2.set_xlabel("Time", fontsize=16)
ax2.set_ylabel("Pairwise Distances", fontsize=16)
ax2.set_title("Total Pairwise Distance vs Time", fontsize=18)
ax2.tick_params(labelsize=13)
ax2.grid(True)
fig2.tight_layout()
plt.show()

## Export Data (optional)

Plots are inline only and are not written to disk. The numeric results
(adjacency matrix + pairwise distances) are still exported below, to
`/content/output/` in the Colab VM. Mount Google Drive first if you want
these files to persist after the runtime disconnects.

In [ ]:
output_dir = "/content/output"
os.makedirs(output_dir, exist_ok=True)

pd.DataFrame(A).to_csv(os.path.join(output_dir, "adjacency_matrix.csv"), index=False)
print("📂 Saved adjacency_matrix.csv")

df = pd.DataFrame({"Time": t})
for i in range(1, N):
    df[f"d_{i}_{i + 1}"] = pairwise_distances[:, i - 1]

df.to_excel(
    os.path.join(output_dir, "pairwise_distances.xlsx"),
    sheet_name="PairwiseDistances",
    index=False,
)
print("📊 Saved pairwise_distances.xlsx")

avg_final_distance = pairwise_distances[-1, :].mean()
print(f"📊 Average final pairwise distance = {round(avg_final_distance, 6)}")

print("✅ COMPLETE")